In [1]:
!pip install flaml[automl] matplotlib openml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.5 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=498fff2372f1c09d289ad9b0b1ce5964cbdd964448c6ccba65aae5588721af1d
  Stored in directory: /root/.cache/pip/wheels/00/23/31/5e562fce1f95aabe57f2a7320d07433ba1cd152bcde2f6a002
Successfully built liac-arff


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from flaml import AutoML

from flaml.automl.model import LGBMEstimator

In [4]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [5]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv').drop(columns=['id'])
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
train['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['humidity'].to_numpy()])
train['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])
train['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['wind_speed'].to_numpy()])

test['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['humidity'].to_numpy()])
test['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])
test['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['wind_speed'].to_numpy()])

column = list(train.columns)
column.remove('efficiency')
column.remove('string_id')
column.remove('error_code')
column.remove('installation_type')

for i_ in range(len(column)):
    for j in column[i_:]:
        i = column[i_]
        if i != j:
            print(f'{i} x {j}')
            train[f'{i} x {j}'] = (train[i].astype('float').to_numpy() * train[j].astype('float').to_numpy())
            test[f'{i} x {j}'] = (test[i].astype('float').to_numpy() * test[j].astype('float').to_numpy())

temperature x irradiance
temperature x humidity
temperature x panel_age
temperature x maintenance_count
temperature x soiling_ratio
temperature x voltage
temperature x current
temperature x module_temperature
temperature x cloud_coverage
temperature x wind_speed
temperature x pressure
irradiance x humidity
irradiance x panel_age
irradiance x maintenance_count
irradiance x soiling_ratio
irradiance x voltage
irradiance x current
irradiance x module_temperature
irradiance x cloud_coverage
irradiance x wind_speed
irradiance x pressure
humidity x panel_age
humidity x maintenance_count
humidity x soiling_ratio
humidity x voltage
humidity x current
humidity x module_temperature
humidity x cloud_coverage
humidity x wind_speed
humidity x pressure
panel_age x maintenance_count
panel_age x soiling_ratio
panel_age x voltage
panel_age x current
panel_age x module_temperature
panel_age x cloud_coverage
panel_age x wind_speed
panel_age x pressure
maintenance_count x soiling_ratio
maintenance_count x 

In [7]:
automl = AutoML()

settings = {
    "time_budget": 2000,
    "metric": "mae",
    "estimator_list": ["lgbm"],
    "task": "regression",
    "log_file_name": "experiment.log",
    "seed": 41,
    "eval_method": "cv"
}
automl.fit(train.drop(columns=['efficiency']), train["efficiency"], **settings)

[flaml.automl.logger: 06-06 03:32:44] {1752} INFO - task = regression
[flaml.automl.logger: 06-06 03:32:44] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 06-06 03:32:44] {1862} INFO - Minimizing error metric: mae
[flaml.automl.logger: 06-06 03:32:44] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 06-06 03:32:44] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 06-06 03:32:45] {2417} INFO - Estimated sufficient time budget=10562s. Estimated necessary time budget=11s.
[flaml.automl.logger: 06-06 03:32:45] {2466} INFO -  at 1.6s,	estimator lgbm's best error=0.0820,	best estimator lgbm's best error=0.0820
[flaml.automl.logger: 06-06 03:32:45] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 06-06 03:32:46] {2466} INFO -  at 2.6s,	estimator lgbm's best error=0.0596,	best estimator lgbm's best error=0.0596
[flaml.automl.logger: 06-06 03:32:46] {2282} INFO - iteration 2, current learner lgbm
[flaml.automl.l

In [8]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = automl.predict(test)
test_predictions

array([0.38521542, 0.52935789, 0.5167876 , ..., 0.6044669 , 0.43608862,
       0.55107812])

In [9]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.385215
1,1,0.529358
2,2,0.516788
3,3,0.477281
4,4,0.469170
...,...,...
11995,11995,0.545864
11996,11996,0.470718
11997,11997,0.604467
11998,11998,0.436089


In [10]:
submission.to_csv('lightautoml.csv', index = False)